In [8]:
import threading

counter = 0
barrier = threading.Barrier(2)

def increment():
    global counter

    current = counter       # Both threads read 0
    barrier.wait()          # Wait until both have read
    counter = current + 1   # Both write 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(counter)  # 1, not 2

1


The Barrier makes both threads reach the same point before continuing.

In [9]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter

    for _ in range(100_000):
        with lock:
            counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print("Counter:", counter)

Counter: 200000


In [10]:
import threading
import time

def task(name):
    print(name, "started")
    time.sleep(2)  # Simulates waiting for I/O
    print(name, "finished")

t1 = threading.Thread(target=task, args=("Thread 1",))
t2 = threading.Thread(target=task, args=("Thread 2",))

start = time.perf_counter()

t1.start()
t2.start()

t1.join()
t2.join()

print("Total:", time.perf_counter() - start)

Thread 1 started
Thread 2 started
Thread 2 finished
Thread 1 finished
Total: 2.008119500009343


Expected execution:
- Thread 1 starts and waits for 2 seconds.
- Thread 2 starts and waits for 2 seconds.
- Both waits overlap.
- Total time is approximately 2 seconds, not 4 seconds.
During sleep(), the thread releases the GIL, allowing another thread to execute. Many real I/O operations similarly release the GIL while waiting.